# Developer scores

Scores each developer from a `gitlogstats` statistics CSV (e.g. `project-2-stats.csv`), where each row is one developer's activity in one repository.

1. **Combine** every row belonging to the same developer (same GitHub username, or the same git name when there is no GitHub username) by summing their metrics, then add the combined metrics in `COMBINED_METRICS` (e.g. `prs_closed` = `prs_merged` + `prs_closed_unmerged`).
2. **Cap** each scored metric at the value in `CAPS`; anything above the cap counts as the cap. This stops one very large number (e.g. a commit that adds thousands of generated lines) from dominating a score.
3. **Z-score** each capped metric across all developers, so metrics measured in different units (lines vs. pull requests) count equally.
4. **Base score** = the mean of a developer's z-scores, mapped linearly onto 0–100 (lowest developer = 0, highest = 100).
5. **Penalties** for `prs_self_merged`, `pushes_to_default` and `web_commits`: each one subtracts up to its `max_penalty` points, growing linearly with the count until it reaches `full_at`. The final score is never below 0.

All settings are in the next cell; nothing below it needs editing to score a different CSV.

In [ ]:
# ---- Settings -------------------------------------------------------------
# Change these to adjust the scoring; the rest of the notebook reads them.

import os

# CSV produced by gitlogstats, and where to write the scores (relative to
# this notebook's folder). The SCORE_INPUT_CSV and SCORE_OUTPUT_CSV
# environment variables override these; assignment-score.sh sets them.
INPUT_CSV = os.environ.get("SCORE_INPUT_CSV", "project-2-stats.csv")
OUTPUT_CSV = os.environ.get("SCORE_OUTPUT_CSV", "project-2-scores.csv")
# set OUTPUT_CSV to None to skip writing results

# Developers left out of scoring entirely (matched case-insensitively against
# the developer key: GitHub username, or git name when there is none).
EXCLUDE_DEVELOPERS = ["bloombar"]

# Metrics built by summing CSV columns. They can be scored in CAPS like any
# other column; the source columns are only scored if listed there too.
COMBINED_METRICS = {
    "prs_closed": ["prs_merged", "prs_closed_unmerged"],
}

# Scored metrics and their caps. A value above its cap counts as the cap.
# Remove a metric from this dict to leave it out of the score.
CAPS = {
    "commits": 15,
    "insertions": 1000,
    "deletions": 300,
    "files": 30,
    "merges": 8,
    "prs_opened": 6,
    "prs_closed": 6,
}

# Penalty metrics: not part of the average. Each subtracts
#   max_penalty * min(count, full_at) / full_at
# points from the 0-100 base score, e.g. with max_penalty 10 and full_at 3,
# a count of 1 costs 3.3 points and a count of 3 or more costs 10.
PENALTIES = {
    "prs_self_merged": {"max_penalty": 10, "full_at": 3},
    "pushes_to_default": {"max_penalty": 10, "full_at": 3},
    "web_commits": {"max_penalty": 10, "full_at": 3},
}

## Load the data

Reads the CSV and checks up front that every column the settings refer to exists, so a typo in the settings fails here with a clear message rather than later with a confusing one.

In [ ]:
import numpy as np
import pandas as pd

pd.set_option("display.max_rows", 200)

raw = pd.read_csv(INPUT_CSV)

# Work out which CSV columns are needed: every scored or penalty metric that
# comes straight from the CSV, plus the columns that combined metrics add up.
source_columns = [c for cols in COMBINED_METRICS.values() for c in cols]
csv_metrics = [c for c in [*CAPS, *PENALTIES] if c not in COMBINED_METRICS]
csv_metrics += [c for c in source_columns if c not in csv_metrics]

missing = [c for c in csv_metrics if c not in raw.columns]
assert not missing, f"columns not in {INPUT_CSV}: {missing}"
print(f"{len(raw)} rows from {INPUT_CSV}")

## 1. Combine rows per developer

A developer can appear in several rows: once per repository, and sometimes under more than one git name. Rows are matched by a **developer key**: the GitHub username, lower-cased (GitHub names are case-insensitive), or the git `username` for rows with no GitHub account. All rows with the same key are added together into one row per developer.

In [ ]:
df = raw.copy()

# Developer key: GitHub username if present, otherwise git username;
# lower-cased and trimmed so "Alice" and "alice " match.
df["developer"] = (
    df["github_username"].fillna(df["username"]).astype(str).str.strip().str.lower()
)

# Blank metric cells mean "none", so count them as 0 rather than missing.
df[csv_metrics] = df[csv_metrics].fillna(0)


def join_unique(values):
    """Join the distinct non-empty values of a column with '; '."""
    return "; ".join(sorted({str(v) for v in values.dropna() if str(v).strip()}))


# One row per developer: list the names and repositories seen, count the rows
# merged, and sum every metric.
devs = df.groupby("developer").agg(
    names=("username", join_unique),
    repositories=("repository", join_unique),
    rows=("repository", "size"),
    **{m: (m, "sum") for m in csv_metrics},
)

# Add each combined metric as the sum of its source columns.
for name, columns in COMBINED_METRICS.items():
    devs[name] = devs[columns].sum(axis=1)

# Drop excluded developers before any statistics are computed, so they do not
# shift the averages that everyone else is compared against.
excluded = devs.index.isin([d.lower() for d in EXCLUDE_DEVELOPERS])
print(f"{len(devs)} developers; excluding {list(devs.index[excluded])}")
devs = devs[~excluded]

# Show the developers whose rows were merged, to check the matching.
devs[devs["rows"] > 1][["names", "repositories", "rows"]]

## 2. Cap the scored metrics

Every value above its metric's cap is replaced by the cap. The table shows how many developers each cap affects; if a cap affects most developers, it is probably too low.

In [ ]:
# clip() replaces each column's values above its cap with the cap.
capped = devs[list(CAPS)].clip(upper=pd.Series(CAPS), axis=1)

pd.DataFrame(
    {
        "cap": pd.Series(CAPS),
        "developers_capped": (devs[list(CAPS)] > pd.Series(CAPS)).sum(),
        "max_before_cap": devs[list(CAPS)].max(),
    }
)

## 3. Z-scores

A z-score says how many standard deviations a developer is above (positive) or below (negative) the average developer for that metric: `z = (value - mean) / standard deviation`. This puts every metric on the same scale, so that e.g. 1000 inserted lines and 6 pull requests can be averaged fairly.

Population z-scores (`ddof=0`) are used. A metric with no variation after capping (everyone has the same value) has z = 0 for everyone, so it neither helps nor hurts.

In [ ]:
# Standard deviation of each metric; a zero (no variation) becomes NaN so
# the division below gives NaN instead of infinity, then fillna(0) sets it to 0.
std = capped.std(ddof=0).replace(0, np.nan)
z = ((capped - capped.mean()) / std).fillna(0)

# Sanity check: each metric's z-scores should have mean ~0.
z.describe().T[["mean", "std", "min", "max"]]

## 4. Base score (0–100)

Each developer's z-scores are averaged, then rescaled linearly so the lowest average becomes 0 and the highest becomes 100. Scores are therefore relative to this group of developers, not absolute.

In [ ]:
mean_z = z.mean(axis=1)

# Min-max rescale to 0-100. If every developer has the same average there is
# nothing to rank, so everyone gets 50.
span = mean_z.max() - mean_z.min()
base_score = (mean_z - mean_z.min()) / span * 100 if span else mean_z * 0 + 50
base_score.describe()

## 5. Penalties

Each penalty metric costs points in proportion to its count, up to that metric's `max_penalty` once the count reaches `full_at`. Penalties are in score points (out of 100) and are added up across metrics. The table lists only developers with a penalty.

In [ ]:
# Penalty per metric = max_penalty * (count, capped at full_at) / full_at.
penalties = pd.DataFrame(
    {
        f"{m}_penalty": devs[m].clip(upper=p["full_at"]) / p["full_at"] * p["max_penalty"]
        for m, p in PENALTIES.items()
    }
)
penalties["total_penalty"] = penalties.sum(axis=1)
penalties[penalties["total_penalty"] > 0].join(devs[list(PENALTIES)])

## 6. Final scores

Final score = base score − total penalty, floored at 0. The results table keeps each developer's uncapped metric totals alongside the intermediate values, so any score can be traced back to its inputs. It is sorted from highest score to lowest and written to `OUTPUT_CSV`.

In [ ]:
scores = devs[["names", "repositories"]].join(
    [
        devs[list(CAPS) + list(PENALTIES)],  # uncapped totals, for reference
        mean_z.rename("mean_z"),
        base_score.rename("base_score"),
        penalties["total_penalty"],
    ]
)
scores["score"] = (scores["base_score"] - scores["total_penalty"]).clip(lower=0)
scores = scores.sort_values("score", ascending=False).round(
    {"mean_z": 3, "base_score": 1, "total_penalty": 1, "score": 1}
)

if OUTPUT_CSV:
    scores.to_csv(OUTPUT_CSV)
    print(f"wrote {len(scores)} developers to {OUTPUT_CSV}")

scores[["names", "base_score", "total_penalty", "score"]]